# Audience Segmentation Analysis

Psychometric segmentation of consumer survey data using K-Means clustering.

Respondents are classified into audience groups based on **values, attitudes, and behavioral patterns** rather than demographics alone.

**Note:** All data is synthetic. No real respondents or client data are included.

| Step | Content |
|---|---|
| 1 | Setup |
| 2 | Synthetic dataset |
| 3 | Exploratory analysis |
| 4 | Feature scaling |
| 5 | Optimal k (Elbow + Silhouette) |
| 6 | K-Means clustering |
| 7 | Segment labeling |
| 8 | Attitude profiling |
| 9 | Demographic profiling |
| 10 | Radar chart |
| 11 | PCA segment map |
| 12 | Segment size chart |
| 13 | Segment summaries |
| 14 | Export |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', None)
np.random.seed(42)

PSYCHOMETRIC_VARS = [
    'sustainability_concern',
    'price_sensitivity',
    'brand_loyalty',
    'digital_engagement',
    'status_orientation',
    'convenience_preference',
    'quality_focus',
    'tradition_values',
]

SEGMENT_COLORS = ['#1D9E75', '#378ADD', '#D4537E', '#BA7517']

print('Setup complete.')

In [ ]:
def make_segment(n, means, std=0.8):
    data = {}
    for var, mean in zip(PSYCHOMETRIC_VARS, means):
        values = np.random.normal(mean, std, n)
        data[var] = np.clip(values, 1, 5).round(0).astype(int)
    return pd.DataFrame(data)

profiles = [
    [4.5, 2.0, 3.5, 3.5, 2.5, 2.5, 4.5, 2.0],
    [2.5, 4.5, 2.0, 3.0, 2.0, 4.5, 2.5, 3.0],
    [3.0, 2.5, 4.0, 4.5, 4.5, 3.5, 3.5, 1.5],
    [2.0, 3.0, 3.0, 1.5, 1.5, 3.0, 3.0, 4.5],
]
sizes = [280, 320, 240, 260]

segments_raw = [make_segment(n, p) for n, p in zip(sizes, profiles)]
df = pd.concat(segments_raw, ignore_index=True)
df['true_segment'] = np.repeat([0,1,2,3], sizes)

n_total = len(df)
df['age']    = np.random.choice([1,2,3,4], n_total, p=[0.2,0.3,0.3,0.2])
df['gender'] = np.random.choice([1,2], n_total, p=[0.48,0.52])
df['income'] = np.random.choice([1,2,3,4,5], n_total, p=[0.15,0.25,0.30,0.20,0.10])
df['region'] = np.random.choice(['North','Northeast','Center-West','Southeast','South'],
                                  n_total, p=[0.08,0.27,0.08,0.42,0.15])

df.loc[df['true_segment']==0,'income'] = np.random.choice([3,4,5], sizes[0], p=[0.3,0.4,0.3])
df.loc[df['true_segment']==1,'income'] = np.random.choice([1,2,3], sizes[1], p=[0.3,0.4,0.3])
df.loc[df['true_segment']==2,'age']    = np.random.choice([1,2,3], sizes[2], p=[0.35,0.40,0.25])
df.loc[df['true_segment']==3,'age']    = np.random.choice([3,4],   sizes[3], p=[0.40,0.60])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset: {len(df)} respondents, {len(PSYCHOMETRIC_VARS)} psychometric variables')
df[PSYCHOMETRIC_VARS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, var in enumerate(PSYCHOMETRIC_VARS):
    counts = df[var].value_counts().sort_index()
    axes[i].bar(counts.index, counts.values, color='#378ADD', alpha=0.85, edgecolor='white')
    axes[i].set_title(var.replace('_',' ').title(), fontsize=10)
    axes[i].set_xlabel('Score (1-5)')
    axes[i].set_xticks([1,2,3,4,5])

plt.suptitle('Distribution of psychometric variables', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[PSYCHOMETRIC_VARS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=[v.replace('_',' ') for v in PSYCHOMETRIC_VARS],
            yticklabels=[v.replace('_',' ') for v in PSYCHOMETRIC_VARS])
ax.set_title('Correlation matrix -- psychometric variables', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[PSYCHOMETRIC_VARS])
X_df = pd.DataFrame(X, columns=PSYCHOMETRIC_VARS)

print(f'Scaled matrix shape: {X_df.shape}')
print(f'Mean after scaling (should be ~0):')
print(X_df.mean().round(3))
print(f'Std after scaling (should be ~1):')
print(X_df.std().round(3))

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, 'o-', color='#378ADD', linewidth=2, markersize=7)
axes[0].axvline(4, color='#E24B4A', linewidth=1.5, linestyle='--', label='k=4 (selected)')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].legend()
axes[0].set_xticks(list(K_range))

axes[1].plot(list(K_range), silhouettes, 'o-', color='#1D9E75', linewidth=2, markersize=7)
axes[1].axvline(4, color='#E24B4A', linewidth=1.5, linestyle='--', label='k=4 (selected)')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()
axes[1].set_xticks(list(K_range))

plt.suptitle('Optimal number of clusters', fontsize=12)
plt.tight_layout()
plt.show()

best_k = list(K_range)[silhouettes.index(max(silhouettes))]
print(f'Best silhouette score: {max(silhouettes):.3f} at k={best_k}')
print('Selected k=4 (statistical + interpretability criteria)')

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=20)
df['segment'] = km.fit_predict(X)

centroids = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_),
                          columns=PSYCHOMETRIC_VARS)

print('Cluster centroids (original scale):')
print(centroids.round(2).to_string())
print(f'\nSilhouette score (k=4): {silhouette_score(X, df["segment"]):.3f}')
print('\nSegment sizes:')
print(df['segment'].value_counts().sort_index())

In [ ]:
label_map = {}
for i, row in centroids.iterrows():
    if row['sustainability_concern'] == centroids['sustainability_concern'].max():
        label_map[i] = 'Conscious Consumer'
    elif row['price_sensitivity'] == centroids['price_sensitivity'].max():
        label_map[i] = 'Practical Consumer'
    elif row['status_orientation'] == centroids['status_orientation'].max():
        label_map[i] = 'Aspirational Consumer'
    else:
        label_map[i] = 'Traditional Consumer'

df['segment_label'] = df['segment'].map(label_map)

print('Label mapping:', label_map)
print('\nSegment distribution:')
print(df['segment_label'].value_counts())

In [ ]:
profile = df.groupby('segment_label')[PSYCHOMETRIC_VARS].mean().round(2)
print('Mean attitude scores by segment (scale 1-5):')
profile

In [ ]:
df['age_label']    = df['age'].map({1:'18-24', 2:'25-34', 3:'35-49', 4:'50+'})
df['gender_label'] = df['gender'].map({1:'Male', 2:'Female'})
df['income_label'] = df['income'].map({1:'Very low', 2:'Low', 3:'Medium', 4:'High', 5:'Very high'})

demo_profile = df.groupby('segment_label').agg(
    N=('segment_label','count'),
    Pct_female=('gender', lambda x: (x==2).mean()*100),
    Avg_age_group=('age','mean'),
    Avg_income_group=('income','mean'),
).round(1)

print('Demographic profile by segment:')
demo_profile

In [ ]:
segment_order = ['Conscious Consumer','Practical Consumer','Aspirational Consumer','Traditional Consumer']
colors = SEGMENT_COLORS

labels_radar = [v.replace('_',' ').title() for v in PSYCHOMETRIC_VARS]
num_vars = len(labels_radar)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 2, figsize=(11, 9), subplot_kw=dict(polar=True))
axes = axes.flatten()

for ax, seg, color in zip(axes, segment_order, colors):
    values = profile.loc[seg, PSYCHOMETRIC_VARS].tolist()
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_radar, fontsize=8)
    ax.set_ylim(1, 5)
    ax.set_yticks([2,3,4,5])
    ax.set_yticklabels(['2','3','4','5'], fontsize=7)
    ax.set_title(seg, fontsize=11, pad=14, color=color)

plt.suptitle('Attitude profiles by segment', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
for seg, color in zip(segment_order, colors):
    mask = df['segment_label'] == seg
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, alpha=0.45, s=25, label=seg, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Segment map -- PCA projection', fontsize=12)
ax.legend(title='Segment', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
total = len(df)
sizes_seg = df['segment_label'].value_counts().reindex(segment_order)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(sizes_seg.index, sizes_seg.values, color=colors, edgecolor='white', height=0.55)
ax.set_xlabel('Number of respondents')
ax.set_title('Segment sizes', fontsize=12)
ax.invert_yaxis()

for bar, val in zip(bars, sizes_seg.values):
    ax.text(bar.get_width()+3, bar.get_y()+bar.get_height()/2,
            f'{val} ({val/total*100:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
summaries = {
    'Conscious Consumer': (
        'Highly concerned with sustainability and quality. '
        'Makes deliberate purchasing decisions, willing to pay more for responsible brands. '
        'Responds well to transparency and values-based communication.'
    ),
    'Practical Consumer': (
        'Price-driven and convenience-oriented. '
        'Low brand loyalty -- will switch easily for a better deal. '
        'Responds well to promotions, clear value propositions and frictionless experience.'
    ),
    'Aspirational Consumer': (
        'Status and trend-oriented, highly engaged with digital content. '
        'Younger skew, motivated by social visibility and brand prestige. '
        'Responds well to influencer marketing and premium positioning.'
    ),
    'Traditional Consumer': (
        'Habitual and routine-driven, low digital engagement. '
        'Older skew, trusts familiar brands and word-of-mouth. '
        'Responds well to reassurance, reliability messaging and in-person channels.'
    ),
}

for seg, color in zip(segment_order, colors):
    n = df['segment_label'].value_counts()[seg]
    pct = n / total * 100
    print(f'{seg} ({n} respondents, {pct:.1f}%)')
    print(f'  {summaries[seg]}')
    print()

In [ ]:
output = df[['segment','segment_label'] + PSYCHOMETRIC_VARS + ['age','gender','income','region']].copy()
output.to_csv('segment_profiles.csv', index=False)
print(f'Saved: segment_profiles.csv')
print(f'Records: {len(output)}')
output.head(10)